# SENDA Copilot · ajuste fino con LoRA (gratis en Google Colab)

Entrena **Qwen3 1.7B** para que sea el copiloto de SENDA: elegir la herramienta correcta, entender
las órdenes (tiempo, tormenta, objetivos, perfil), explicar cómo se usa el sistema y responder
solo con las cifras del planificador.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

Pasos: instalar → cargar datos → cargar modelo → LoRA → entrenar → probar → exportar a GGUF.
Tiempo aproximado en una T4: 45–90 min de entrenamiento + 10 min de exportación.

Los datos los genera `backend/training/generate_dataset.py` con el planificador real
(`train.jsonl` y `val.jsonl`). El examen `backend/eval/run_eval.py` mide antes y después.

## 1. Instalar Unsloth

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo

## 2. Cargar los datos

Opción A: sube `train.jsonl` y `val.jsonl` (carpeta `backend/training/data/`).
Opción B: si el repositorio es público, descárgalos con la URL raw de GitHub.

In [ ]:
import os
REPO_RAW = ""  # p. ej. "https://raw.githubusercontent.com/<usuario>/senda-mars/<rama>/backend/training/data"

if REPO_RAW:
    !wget -q {REPO_RAW}/train.jsonl -O train.jsonl
    !wget -q {REPO_RAW}/val.jsonl -O val.jsonl
else:
    from google.colab import files
    print("Sube train.jsonl y val.jsonl")
    files.upload()

assert os.path.exists("train.jsonl") and os.path.exists("val.jsonl"), "Faltan los archivos de datos"
!wc -l train.jsonl val.jsonl

## 3. Cargar Qwen3 1.7B en 4 bits

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 4096   # prompt de sistema + herramientas + documentación + respuesta
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)

## 4. Adaptadores LoRA

Solo se entrena ~1–2 % de los pesos: cabe en la T4 y conserva lo que el modelo ya sabe.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)

## 5. Formatear con la plantilla de chat de Qwen3

Mismo formato que usa Ollama en producción: herramientas en el prompt de sistema,
`<tool_call>` para las llamadas y `<tool_response>` para los resultados. Sin modo "thinking"
(el backend llama con `think: false`).

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})

def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tools=example["tools"], tokenize=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    return {"text": text, "n_tokens": len(tokenizer(text).input_ids)}

data = data.map(to_text, remove_columns=[c for c in data["train"].column_names if c != "meta"])
too_long = data["train"].filter(lambda x: x["n_tokens"] > MAX_SEQ)
print("ejemplos:", len(data["train"]), "train /", len(data["val"]), "val · demasiado largos:", len(too_long))
data = data.filter(lambda x: x["n_tokens"] <= MAX_SEQ)
print("tokens: media", sum(data["train"]["n_tokens"]) // len(data["train"]), "· máx", max(data["train"]["n_tokens"]))
print(data["train"][0]["text"][-1200:])

## 6. Entrenar

`train_on_responses_only` hace que el modelo aprenda **solo sus propios turnos** (las llamadas a
herramientas y las respuestas). El prompt, la orden del usuario y los resultados del planificador
(que en Qwen3 van dentro de un turno `user` como `<tool_response>`) no se aprenden: se leen.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

EPOCHS = 2

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data["train"],
    eval_dataset=data["val"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="no",
        optim="adamw_8bit",
        seed=7,
        report_to="none",
        output_dir="outputs",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

## 7. Probar antes de exportar

In [ ]:
import json
FastLanguageModel.for_inference(model)

sample = json.loads(open("val.jsonl", encoding="utf-8").readline())
system, tools = sample["messages"][0], sample["tools"]

def ask(question):
    msgs = [system, {"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, temperature=0.2, do_sample=True)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=False)

for q in ["Arma la misión de hoy, tenemos dos horas", "¿Cómo pongo una zona de peligro?",
          "How much oxygen is left on the walk?", "¿Cuál es la capital de Francia?"]:
    print("›", q, "\n", ask(q), "\n")

## 8. Exportar a GGUF (para Ollama)

Fusiona los adaptadores con el modelo y lo cuantiza a `q4_k_m` (~1.1 GB), el mismo formato del
modelo base. Al terminar, descarga el archivo `.gguf`.

In [ ]:
model.save_pretrained_gguf("senda-copilot", tokenizer, quantization_method="q4_k_m")
!ls -lh senda-copilot*/*.gguf senda-copilot*.gguf 2>/dev/null

In [ ]:
import glob
from google.colab import files
gguf = sorted(glob.glob("senda-copilot*/*.gguf") + glob.glob("senda-copilot*.gguf"))
print(gguf)
files.download([g for g in gguf if "q4_k_m" in g.lower()][0])

## 9. En tu PC

```powershell
cd backend
python training/install_in_ollama.py ruta\al\senda-copilot.Q4_K_M.gguf   # crea el modelo "senda-copilot"
python eval/run_eval.py --model senda-copilot                            # examen: compara con qwen3:1.7b
```

Si la nota mejora, en `backend/.env` pon `SENDA_LLM_MODEL=senda-copilot` y reinicia el backend.